# 🎙️ Chatterbox Indonesian Zero-Shot Voice Cloning — Colab Quality & Benchmark Test
### TaSTP YouTube Dubbing Studio — Phase 1 Quality Test Notebook

Notebook ini dirancang untuk menguji model **`grandhigh/Chatterbox-TTS-Indonesian`** (Apache-2.0, fine-tune dari `ResembleAI/chatterbox` MIT) pada GPU Google Colab (T4 / V100 / A100).

#### 🎯 Tujuan Pengujian (Phase 1):
1. **Grid Parameter Test:** Matriks `exaggeration` (0.3, 0.5, 0.7, 0.9) × `cfg_weight` (0.0, 0.3, 0.5) pada 3 sampel suara referensi (Netral, Ceria, Serius).
2. **Panjang Referensi Audio:** Menguji durasi referensi 5 detik, 10 detik, dan 20 detik.
3. **Ketahanan Noise:** Membandingkan referensi audio bersih vs referensi dengan background noise.
4. **Uji Naskah Panjang:** Mengevaluasi stabilitas dan risiko halusinasi/pengulangan kata pada paragraf narasi YouTube panjang (100+ kata).
5. **Variasi Seed:** Menguji konsistensi timbre dan melodi vokal antar seed acak.
6. **Metrik Objektif:** Mengukur durasi audio, latency sintesis, alokasi VRAM GPU, Mean F0, Std F0 (indikator dinamika prosodi), dan RMS.

## 1. Verifikasi GPU & Pemasangan Dependensi
> Pastikan runtime Colab diatur ke **T4 GPU** (`Runtime` -> `Change runtime type` -> `T4 GPU`).

In [ ]:
# 1. Periksa Akselerasi Hardware GPU
!nvidia-smi

# 2. Pasang dependensi Chatterbox TTS sesuai rekomendasi resmi model card
# Pinning transformers==4.44.2 dan peft==0.11.1 untuk mencegah konflik versi safetensors/attention
!pip install -q chatterbox-tts==0.1.1 transformers==4.44.2 peft==0.11.1
!pip install -q -U ml_dtypes soundfile torchaudio scipy numpy librosa

import os
import time
import torch
import torchaudio as ta
import soundfile as sf
import numpy as np
from pathlib import Path

print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name: {torch.cuda.get_device_name(0)}")
    print(f"Device Memory: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")

## 2. Memuat Model Dasar Chatterbox & Checkpoint Fine-Tune Bahasa Indonesia (`grandhigh/Chatterbox-TTS-Indonesian`)

In [ ]:
from chatterbox.tts import ChatterboxTTS
from huggingface_hub import hf_hub_download
from safetensors.torch import load_file

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Loading Chatterbox base model on device: {device}...")

t0 = time.time()
# Inisialisasi arsitektur Chatterbox TTS
model = ChatterboxTTS.from_pretrained(device=device)

# Unduh dan pasang bobot fine-tuned Bahasa Indonesia (t3_cfg.safetensors)
MODEL_REPO = "grandhigh/Chatterbox-TTS-Indonesian"
CHECKPOINT_FILENAME = "t3_cfg.safetensors"

print(f"Downloading checkpoint '{CHECKPOINT_FILENAME}' from {MODEL_REPO}...")
checkpoint_path = hf_hub_download(repo_id=MODEL_REPO, filename=CHECKPOINT_FILENAME)

print("Applying Indonesian fine-tune weights to T3 transformer...")
t3_state = load_file(checkpoint_path)
model.t3.load_state_dict(t3_state)

load_time = time.time() - t0
print(f"✅ Model Chatterbox Indonesian berhasil dimuat dalam {load_time:.2f} detik!")
print(f"Sample Rate Output: {model.sr} Hz")

## 3. Siapkan Audio Referensi Suara (Voice Prompts)
Anda dapat mengunggah 3 rekaman suara asli Anda (format WAV/MP3 mono, durasi 10–20 detik) dengan karakter emosi:
1. `ref_neutral.wav` : Gaya santai, percakapan normal sehari-hari.
2. `ref_cheerful.wav`: Gaya antusias, ceria, tempo berenergi untuk intro/vlog.
3. `ref_serious.wav` : Gaya tegas, berwibawa, nada turun untuk penjelasan penting.

*Jika Anda belum mengunggah rekaman pribadi, sel di bawah ini akan secara otomatis membuat audio referensi dummy agar seluruh alur pengujian dapat dijalankan langsung.*

In [ ]:
REF_DIR = Path("reference_voices")
REF_DIR.mkdir(parents=True, exist_ok=True)

# Fungsi bantu membuat audio referensi sintetis jika file belum ada
def ensure_reference_audio(ref_path: Path, base_freq: float = 180.0, duration_sec: float = 12.0):
    if ref_path.exists():
        print(f"  [OK] Menggunakan file referensi asli: {ref_path} ({ref_path.stat().st_size / 1024:.1f} KB)")
        return
    
    sr = 24000
    t = np.linspace(0, duration_sec, int(sr * duration_sec), endpoint=False)
    # Harmonik vokal sederhana (f0 + f1 + f2)
    sig = (0.5 * np.sin(2 * np.pi * base_freq * t) +
           0.25 * np.sin(2 * np.pi * (base_freq * 2) * t) +
           0.15 * np.sin(2 * np.pi * (base_freq * 3) * t))
    # Modulasi volume natural
    envelope = 0.5 * (1.0 + np.sin(2 * np.pi * 0.5 * t))
    audio = (sig * envelope).astype(np.float32)
    sf.write(str(ref_path), audio, sr)
    print(f"  [DUMMY] Dibuat referensi dummy sementara: {ref_path}")

ref_neutral = REF_DIR / "ref_neutral.wav"
ref_cheerful = REF_DIR / "ref_cheerful.wav"
ref_serious = REF_DIR / "ref_serious.wav"

ensure_reference_audio(ref_neutral, base_freq=165.0, duration_sec=12.0)
ensure_reference_audio(ref_cheerful, base_freq=210.0, duration_sec=12.0)
ensure_reference_audio(ref_serious, base_freq=135.0, duration_sec=12.0)

## 4. Fungsi Analisis Metrik Akustik & Diagnostik Suara
Metrik yang diukur secara presisi:
- **Durasi (detik)**
- **Latency (detik)**: Waktu komputasi sintesis per kalimat pada GPU.
- **Peak VRAM (MB)**: Memori GPU maksimum yang terkonsumsi.
- **Mean F0 (Hz)**: Rata-rata frekuensi dasar nada vokal.
- **Std F0 (Hz)**: Standar deviasi F0 (indikator melodi/ekspresi; nilai rendah = datar/monoton, nilai tinggi = sangat dinamis).
- **RMS**: Tingkat energi intensitas suara.

In [ ]:
def extract_audio_metrics(wav_tensor: torch.Tensor, sr: int, latency_sec: float, peak_vram_mb: float):
    samples = wav_tensor.squeeze().cpu().numpy().astype(np.float32)
    if len(samples) == 0:
        return {"duration": 0.0, "latency": latency_sec, "vram_mb": peak_vram_mb, "mean_f0": 0.0, "std_f0": 0.0, "rms": 0.0}
    
    duration = len(samples) / float(sr)
    rms = float(np.sqrt(np.mean(samples ** 2)))
    
    # Frame-based autocorrelation untuk pitch tracking F0 vokal manusia (75 Hz - 450 Hz)
    frame_len = int(sr * 0.030) # 30 ms
    hop_len = int(sr * 0.010)   # 10 ms
    min_lag = int(sr / 450.0)
    max_lag = int(sr / 75.0)
    
    f0_list = []
    for i in range(0, len(samples) - frame_len, hop_len):
        frame = samples[i : i + frame_len]
        if np.max(np.abs(frame)) < 0.01:
            continue
        win = np.hamming(len(frame))
        frame_win = frame * win
        corr = np.correlate(frame_win, frame_win, mode='full')
        corr = corr[len(corr) // 2 :]
        if len(corr) < max_lag:
            continue
        corr_slice = corr[min_lag:max_lag]
        if len(corr_slice) == 0:
            continue
        peak_idx = min_lag + np.argmax(corr_slice)
        if corr[0] > 0 and (corr[peak_idx] / corr[0]) > 0.35:
            f0 = sr / float(peak_idx)
            if 75.0 <= f0 <= 450.0:
                f0_list.append(f0)
    
    mean_f0 = float(np.mean(f0_list)) if f0_list else 0.0
    std_f0 = float(np.std(f0_list)) if f0_list else 0.0
    
    return {
        "duration": round(duration, 2),
        "latency": round(latency_sec, 2),
        "vram_mb": round(peak_vram_mb, 1),
        "mean_f0": round(mean_f0, 1),
        "std_f0": round(std_f0, 1),
        "rms": round(rms, 4),
    }

## 5. UJI UTAMA A: Grid Test Parameter Expressiveness
- **Kalimat Uji (3 Karakter Narasi YouTube):**
  1. *Kalimat 1 (Intro/Narasi Umum):* `"Halo semuanya, selamat datang kembali di channel ini, hari ini kita akan membahas topik yang sangat menarik."`
  2. *Kalimat 2 (Penjelasan Teknis/Detail):* `"Proses ini membutuhkan ketelitian tinggi agar hasil suaranya terdengar jernih, natural, dan sinkron dengan video."`
  3. *Kalimat 3 (Call-to-Action/Antusias):* `"Jangan lupa untuk tekan tombol like, tinggalkan komentar di bawah, dan subscribe agar tidak ketinggalan video terbaru!"`
- **Parameter Grid:**
  - `exaggeration`: `[0.3, 0.5, 0.7, 0.9]`
  - `cfg_weight`: `[0.0, 0.3, 0.5]`

In [ ]:
OUTPUT_DIR = Path("test_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TEST_SENTENCES = [
    ("s1_intro", "Halo semuanya, selamat datang kembali di channel ini, hari ini kita akan membahas topik yang sangat menarik."),
    ("s2_detail", "Proses ini membutuhkan ketelitian tinggi agar hasil suaranya terdengar jernih, natural, dan sinkron dengan video."),
    ("s3_cta", "Jangan lupa untuk tekan tombol like, tinggalkan komentar di bawah, dan subscribe agar tidak ketinggalan video terbaru!"),
]

REFERENCES = [
    ("neutral", ref_neutral),
    ("cheerful", ref_cheerful),
    ("serious", ref_serious),
]

EXAGGERATIONS = [0.3, 0.5, 0.7, 0.9]
CFG_WEIGHTS = [0.0, 0.3, 0.5]

grid_results = []

print("=" * 95)
print("MEMULAI GRID TEST (Exaggeration × CFG Weight)")
print("=" * 95)

# Lakukan uji coba grid pada kalimat intro lintas referensi vokal
for ref_label, ref_file in REFERENCES:
    for exag in EXAGGERATIONS:
        for cfg in CFG_WEIGHTS:
            sent_id, sent_text = TEST_SENTENCES[0] # s1_intro
            out_filename = f"grid_{ref_label}_{sent_id}_exag{exag}_cfg{cfg}.wav"
            out_path = OUTPUT_DIR / out_filename
            
            if torch.cuda.is_available():
                torch.cuda.reset_peak_memory_stats()
            
            t_start = time.time()
            try:
                wav = model.generate(
                    text=sent_text,
                    audio_prompt_path=str(ref_file),
                    exaggeration=exag,
                    cfg_weight=cfg,
                )
                t_elapsed = time.time() - t_start
                peak_vram = (torch.cuda.max_memory_allocated() / (1024**2)) if torch.cuda.is_available() else 0.0
                
                # Simpan audio WAV
                ta.save(str(out_path), wav.squeeze(0).cpu(), model.sr)
                
                metrics = extract_audio_metrics(wav, model.sr, t_elapsed, peak_vram)
                metrics.update({
                    "test": "Grid",
                    "ref": ref_label,
                    "exag": exag,
                    "cfg": cfg,
                    "file": out_filename,
                })
                grid_results.append(metrics)
                print(f"  [OK] {ref_label:<8} | Exag: {exag} | CFG: {cfg} -> Dur: {metrics['duration']}s | F0: {metrics['mean_f0']}Hz (±{metrics['std_f0']}Hz) | Lat: {metrics['latency']}s | VRAM: {metrics['vram_mb']}MB")
            except Exception as e:
                print(f"  [ERR] {ref_label} | Exag: {exag} | CFG: {cfg} -> {e}")

print("\n✅ Grid Test Selesai!")

## 6. UJI B: Systematic Edge Cases
Pengujian parameter kritis untuk produksi studio dubbing YouTube:
1. **Reference Length (5s vs 10s vs 20s):** Berapa durasi rekaman referensi vokal ideal?
2. **Clean vs Noisy Reference:** Seberapa sensitif model terhadap rekaman mic ruangan ber-noise?
3. **Long Paragraph Test:** Apakah terjadi degradasi suara, halusinasi, atau loop repetisi kata pada narasi panjang?
4. **Seed Variation:** Seberapa konsisten warna suara bila seed acak berbeda?

In [ ]:
edge_results = []
print("=" * 95)
print("MEMULAI UJI B: SYSTEMATIC EDGE CASES")
print("=" * 95)

# --- 1. Uji Durasi Referensi (5s, 10s, 20s) ---
print("\n--- 1. Uji Durasi Sampel Referensi (5s / 10s / 20s) ---")
raw_audio, sr_orig = ta.load(str(ref_neutral))
for dur_target in [5, 10, 20]:
    sample_cut = raw_audio[:, : min(raw_audio.shape[1], sr_orig * dur_target)]
    cut_path = REF_DIR / f"ref_cut_{dur_target}s.wav"
    ta.save(str(cut_path), sample_cut, sr_orig)
    
    t_start = time.time()
    wav = model.generate(
        text=TEST_SENTENCES[0][1],
        audio_prompt_path=str(cut_path),
        exaggeration=0.5,
        cfg_weight=0.3,
    )
    t_elapsed = time.time() - t_start
    peak_vram = (torch.cuda.max_memory_allocated() / (1024**2)) if torch.cuda.is_available() else 0.0
    
    fname = f"edge_reflen_{dur_target}s.wav"
    ta.save(str(OUTPUT_DIR / fname), wav.squeeze(0).cpu(), model.sr)
    m = extract_audio_metrics(wav, model.sr, t_elapsed, peak_vram)
    m.update({"test": f"RefLen_{dur_target}s", "ref": "neutral", "exag": 0.5, "cfg": 0.3, "file": fname})
    edge_results.append(m)
    print(f"  [Ref {dur_target}s] -> Dur: {m['duration']}s | F0: {m['mean_f0']}Hz (±{m['std_f0']}Hz) | Lat: {m['latency']}s")

# --- 2. Uji Ketahanan Terhadap Noise ---
print("\n--- 2. Uji Ketahanan Terhadap Background Noise ---")
noise = torch.randn_like(raw_audio) * 0.05 # Tambahkan noise tiruan (SNR sedang)
noisy_audio = raw_audio + noise
noisy_path = REF_DIR / "ref_noisy.wav"
ta.save(str(noisy_path), noisy_audio, sr_orig)

t_start = time.time()
wav_noisy = model.generate(
    text=TEST_SENTENCES[0][1],
    audio_prompt_path=str(noisy_path),
    exaggeration=0.5,
    cfg_weight=0.3,
)
t_elapsed = time.time() - t_start
fname_noisy = "edge_noisy_reference.wav"
ta.save(str(OUTPUT_DIR / fname_noisy), wav_noisy.squeeze(0).cpu(), model.sr)
m_noisy = extract_audio_metrics(wav_noisy, model.sr, t_elapsed, 0.0)
m_noisy.update({"test": "NoisyRef", "ref": "noisy", "exag": 0.5, "cfg": 0.3, "file": fname_noisy})
edge_results.append(m_noisy)
print(f"  [Noisy Ref] -> Dur: {m_noisy['duration']}s | F0: {m_noisy['mean_f0']}Hz (±{m_noisy['std_f0']}Hz) | RMS: {m_noisy['rms']}")

# --- 3. Uji Paragraf Panjang (Hallucination & Repetition Check) ---
print("\n--- 3. Uji Paragraf Narasi Panjang (100+ Kata) ---")
LONG_PARAGRAPH = (
    "Selamat datang kembali di kanal dokumenter sains dan teknologi masa depan. "
    "Hari ini kita akan mengupas secara mendalam bagaimana revolusi kecerdasan buatan telah mengubah cara kreator konten "
    "memproduksi suara vokal dan narasi video profesional. Di masa lalu, dubbing suara membutuhkan studio rekaman dengan biaya tinggi "
    "serta waktu produksi berhari-hari. Namun kini, dengan teknologi neural voice cloning berbahasa Indonesia, kita dapat mempertahankan "
    "karakter vokal asli kita dengan intonasi yang konsisten dan artikulasi yang sangat presisi di setiap babak cerita video kita."
)
t_start = time.time()
wav_long = model.generate(
    text=LONG_PARAGRAPH,
    audio_prompt_path=str(ref_neutral),
    exaggeration=0.5,
    cfg_weight=0.3,
)
t_elapsed = time.time() - t_start
peak_vram_long = (torch.cuda.max_memory_allocated() / (1024**2)) if torch.cuda.is_available() else 0.0
fname_long = "edge_long_paragraph.wav"
ta.save(str(OUTPUT_DIR / fname_long), wav_long.squeeze(0).cpu(), model.sr)
m_long = extract_audio_metrics(wav_long, model.sr, t_elapsed, peak_vram_long)
m_long.update({"test": "LongParagraph", "ref": "neutral", "exag": 0.5, "cfg": 0.3, "file": fname_long})
edge_results.append(m_long)
print(f"  [Long Paragraph] -> Dur: {m_long['duration']}s | Latency: {m_long['latency']}s | VRAM: {m_long['vram_mb']}MB | F0: {m_long['mean_f0']}Hz")

# --- 4. Uji Variasi Seed Acak ---
print("\n--- 4. Uji Variasi Seed Acak (Konsistensi Timbre) ---")
for seed_val in [42, 123, 999]:
    torch.manual_seed(seed_val)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed_val)
    
    t_start = time.time()
    wav_seed = model.generate(
        text=TEST_SENTENCES[0][1],
        audio_prompt_path=str(ref_neutral),
        exaggeration=0.5,
        cfg_weight=0.3,
    )
    t_elapsed = time.time() - t_start
    fname_seed = f"edge_seed_{seed_val}.wav"
    ta.save(str(OUTPUT_DIR / fname_seed), wav_seed.squeeze(0).cpu(), model.sr)
    m_seed = extract_audio_metrics(wav_seed, model.sr, t_elapsed, 0.0)
    m_seed.update({"test": f"Seed_{seed_val}", "ref": "neutral", "exag": 0.5, "cfg": 0.3, "file": fname_seed})
    edge_results.append(m_seed)
    print(f"  [Seed {seed_val}] -> Dur: {m_seed['duration']}s | F0: {m_seed['mean_f0']}Hz (±{m_seed['std_f0']}Hz)")

print("\n✅ Systematic Edge Cases Selesai!")

## 7. Rangkuman Tabel Metrik & Hasil Evaluasi Kualitas

In [ ]:
import pandas as pd
from IPython.display import display, HTML

all_results = grid_results + edge_results
df = pd.DataFrame(all_results)

print("\n" + "=" * 110)
print("                TABEL LAPORAN PENGUJIAN CHATTERBOX INDONESIAN (PHASE 1)")
print("=" * 110)
print(f"| {'Uji':<14} | {'Ref':<8} | {'Exag':<5} | {'CFG':<5} | {'Dur(s)':<7} | {'Mean F0':<8} | {'Std F0':<8} | {'Lat(s)':<7} | {'VRAM(MB)':<9} | {'File Audio':<26} |")
print("|" + "-"*16 + "|" + "-"*10 + "|" + "-"*7 + "|" + "-"*7 + "|" + "-"*9 + "|" + "-"*10 + "|" + "-"*10 + "|" + "-"*9 + "|" + "-"*11 + "|" + "-"*28 + "|")

for r in all_results:
    print(f"| {r['test']:<14} | {r['ref']:<8} | {r['exag']:<5} | {r['cfg']:<5} | {r['duration']:<7.2f} | {r['mean_f0']:<8.1f} | {r['std_f0']:<8.1f} | {r['latency']:<7.2f} | {r['vram_mb']:<9.1f} | {r['file']:<26} |")
print("=" * 110)

# Simpan laporan ke CSV
df.to_csv("chatterbox_test_report.csv", index=False)
print("\nLaporan lengkap tersimpan di: chatterbox_test_report.csv")

## 8. Pemutar Audio Interaktif & Unduh Seluruh Hasil Uji (.zip)
Gunakan sel ini untuk mendengarkan langsung sampel audio di dalam Google Colab dan mengunduh seluruh file WAV untuk didengarkan secara offline.

In [ ]:
from IPython.display import Audio, display

print("--- 🎧 DENGARKAN SAMPEL REKOMENDASI UTAMA ---")
key_samples = [
    ("Rekomendasi Default (Exag 0.5, CFG 0.3)", OUTPUT_DIR / "grid_neutral_s1_intro_exag0.5_cfg0.3.wav"),
    ("Ekspresif Tinggi (Exag 0.7, CFG 0.3)", OUTPUT_DIR / "grid_cheerful_s1_intro_exag0.7_cfg0.3.wav"),
    ("Paragraf Panjang Narasi (100+ Kata)", OUTPUT_DIR / "edge_long_paragraph.wav"),
    ("Referensi 10 Detik vs 5 Detik", OUTPUT_DIR / "edge_reflen_10s.wav"),
]

for desc, path in key_samples:
    if path.exists():
        print(f"\n▶️ {desc} ({path.name}):")
        display(Audio(str(path)))

# Kemas seluruh output menjadi ZIP agar mudah diunduh
!zip -q -r chatterbox_test_results.zip test_outputs/ reference_voices/ chatterbox_test_report.csv
print("\n📦 Seluruh output berhasil dikemas ke: chatterbox_test_results.zip")
print("Anda dapat mengunduhnya langsung dari panel Files di sebelah kiri Google Colab.")